# Submit readiness notebook

Run this on Kaggle with T4 and the competition SDK/model datasets attached.
Checks whether the current `src/attack.py` family is submit-ready under the
live SDK and writes `research/results/validation-summary.latest.json`,
consumed by `make submit-ready`. This kernel is expected to fail (papermill
error) when the current family is not submit-ready — that is the intended
signal, not a bug.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys


def find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/AI-Agent-Security')]
    for candidate in candidates:
        if (candidate / 'tools' / 'run_gguf_validation.py').exists():
            return candidate
    raise FileNotFoundError('tools/run_gguf_validation.py not found; run from the repo root')


ROOT = find_repo_root()
os.chdir(ROOT)
print('repo root:', ROOT)
gpu = subprocess.run(['nvidia-smi', '-L'], text=True, capture_output=True)
print(gpu.stdout.strip() or gpu.stderr.strip() or 'no GPU visible')

In [ ]:
os.environ.setdefault('GPT_OSS_GGUF_REPO', 'unsloth/gpt-oss-20b-GGUF')
os.environ.setdefault('GPT_OSS_GGUF_FILE', 'gpt-oss-20b-Q4_K_M.gguf')
os.environ.setdefault('GEMMA_GGUF_REPO', 'unsloth/gemma-4-26B-A4B-it-GGUF')
os.environ.setdefault('GEMMA_GGUF_FILE', 'gemma-4-26B-A4B-it-UD-Q4_K_M.gguf')

# For offline Kaggle runs, set these to attached dataset file paths before running:
# os.environ['GPT_OSS_MODEL_PATH'] = '/kaggle/input/<dataset>/gpt-oss-20b-Q4_K_M.gguf'
# os.environ['GEMMA_MODEL_PATH'] = '/kaggle/input/<dataset>/gemma-4-26B-A4B-it-UD-Q4_K_M.gguf'

for name in ('GPT_OSS_MODEL_PATH', 'GEMMA_MODEL_PATH'):
    if os.getenv(name):
        print(name, os.getenv(name))

In [ ]:
import importlib.util


def ensure_llama_cpp() -> None:
    if importlib.util.find_spec('llama_cpp') is not None:
        print('llama_cpp already installed')
        return
    extra_index = os.getenv(
        'LLAMA_CPP_EXTRA_INDEX_URL',
        'https://abetlen.github.io/llama-cpp-python/whl/cu124',
    )
    wheel_cmd = [
        sys.executable,
        '-m', 'pip', 'install', '-q', '--prefer-binary',
        'llama-cpp-python', '--extra-index-url', extra_index,
    ]
    print('installing llama-cpp-python from', extra_index)
    try:
        subprocess.run(wheel_cmd, check=True)
    except subprocess.CalledProcessError:
        print('prebuilt wheel install failed; building llama-cpp-python with CUDA')
        env = os.environ.copy()
        env.setdefault('CMAKE_ARGS', '-DGGML_CUDA=on')
        env.setdefault('FORCE_CMAKE', '1')
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall', 'llama-cpp-python'],
            check=True,
            env=env,
        )
    if importlib.util.find_spec('llama_cpp') is None:
        raise ModuleNotFoundError('llama_cpp')


ensure_llama_cpp()

In [ ]:
cmd = [
    sys.executable,
    'tools/run_gguf_validation.py',
    '--n', os.getenv('VALIDATION_N', '20'),
    '--models', os.getenv('VALIDATION_MODELS', 'gpt_oss,gemma'),
    '--budget-per-model', os.getenv('VALIDATION_BUDGET_PER_MODEL', '3000'),
    '--max-tool-hops', os.getenv('VALIDATION_MAX_TOOL_HOPS', '8'),
    '--env-selection', os.getenv('VALIDATION_ENV_SELECTION', 'gym'),
    '--out', 'research/results/validation-summary.latest.json',
    '--raw-out', 'research/results/validation-raw.latest.jsonl',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
if os.getenv('RUN_SUPPRESS_AB_EXPERIMENT') == '1':
    families = os.getenv(
        'EXPERIMENT_FAMILIES',
        'direct_exfil_suppress_once,direct_exfil_suppress_stop,direct_exfil_suppress_no_summary,direct_exfil_exactly_once,direct_exfil_minimal_function',
    )
    bank_cmd = [
        sys.executable,
        'research/candidate_families.py',
        '--families', families,
        '--n', os.getenv('EXPERIMENT_N', '5'),
        '--out', 'research/results/candidate_bank.suppress_ab.jsonl',
    ]
    print(' '.join(bank_cmd))
    subprocess.run(bank_cmd, check=True)
    experiment_cmd = [
        sys.executable,
        'tools/run_gguf_bank_experiment.py',
        '--bank', 'research/results/candidate_bank.suppress_ab.jsonl',
        '--models', os.getenv('VALIDATION_MODELS', 'gpt_oss,gemma'),
        '--budget-per-model', os.getenv('VALIDATION_BUDGET_PER_MODEL', '3000'),
        '--max-tool-hops', os.getenv('VALIDATION_MAX_TOOL_HOPS', '8'),
        '--env-selection', os.getenv('VALIDATION_ENV_SELECTION', 'gym'),
        '--out', 'research/results/suppress-ab-gguf.latest.json',
        '--raw-out', 'research/results/suppress-ab-gguf.raw.jsonl',
    ]
    print(' '.join(experiment_cmd))
    subprocess.run(experiment_cmd, check=True)
else:
    print('RUN_SUPPRESS_AB_EXPERIMENT is not set; skipping suppressor A/B experiment')

In [ ]:
import json

summary_path = Path('research/results/validation-summary.latest.json')
summary = json.loads(summary_path.read_text())
print(json.dumps({
    'schema_version': summary.get('schema_version'),
    'validation_n': summary.get('validation_n'),
    'max_tool_hops': summary.get('max_tool_hops'),
    'backend': summary.get('backend'),
    'results': summary.get('results'),
}, indent=2, sort_keys=True))